# Step 10.3 — Bahdanau-style Attention Intuition：让 Decoder 每一步都能“回头看” Source

Step 10.2 的 fixed-vector bottleneck 已经把问题暴露得很清楚：encoder 明明产生了整条 `(B,T_src,H)` state trajectory，但经典 Seq2Seq 只把最后一个 state `(B,H)` 交给 decoder。

Bahdanau-style attention 的核心想法是：

> **不要要求 decoder 永远只依赖一个固定 source summary；在每个 decoder step，根据当前生成状态，重新判断哪些 encoder states 更相关。**

本节只建立 intuition：decoder state 怎样提出“现在我需要 source 哪部分信息？”、alignment / attention weights 是什么、为什么 context 可以随 decoder step 改变。

本节不进入 Step 10.4：不推导 softmax，不做 score→softmax→weighted-sum 的精确数值计算，也不使用 Q/K/V 术语。


## 1. 从一个 Final State，改成允许访问整条 Encoder State Trajectory

经典 fixed-vector bridge：

$$
H^{enc}
\in
\mathbb R^{B\times T_{src}\times H}
\quad\longrightarrow\quad
s=h_{T_{src}-1}^{enc}
\in
\mathbb R^{B\times H}.
$$

decoder 后续主要依赖这个 final state。

Attention 的结构变化是：encoder 仍然产生：

$$
h_0^{enc},h_1^{enc},\ldots,h_{T_{src}-1}^{enc}
$$

但这些 intermediate encoder states 不再只作为“内部过程”存在；decoder 的每个生成步骤都可以根据需要重新利用它们。

可以把它想成：

```text
source positions
   0      1      2          T_src-1
   ↓      ↓      ↓             ↓
 h_0    h_1    h_2    ...   h_(T_src-1)
   \      |      |             /
    \     |      |            /
     ---- decoder step u -----
             ↓
      decide what matters now
```

注意：Attention 不等于删除 encoder recurrence。这个历史阶段仍然可以使用 recurrent encoder / decoder；改变的是 decoder 访问 source representations 的方式。


## 2. Decoder State 像一个“问题”：做 Content-Based Lookup

在 decoder 准备生成第 `u` 个 target position 时，它已经有一个 previous decoder state：

$$
h_{u-1}^{dec}
\in
\mathbb R^{B\times H_{dec}}.
$$

这个 state 概括了 decoder 到目前为止已经生成/处理的 target-side context。

现在它要问：

> “为了生成下一个 target token，source 的哪些位置最相关？”

因此对每个 source position `j`，可以让一个 learned **alignment / compatibility function（对齐/兼容度函数）** 比较：

```text
previous decoder state h_(u-1)^dec
             与
encoder state h_j^enc
```

并产生一个 raw relevance value：

$$
e_{u,j}
=
a\!\left(
h_{u-1}^{dec},
h_j^{enc}
\right).
$$

`e_(u,j)` 可以先理解成：

> decoder step `u` 对 source position `j` 的 **raw alignment score（原始对齐分数）**。

这里的 `a(·,·)` 是可学习函数；本节不展开它具体怎样用矩阵、非线性层来实现。

### 为什么叫 Content-Based？

因为 decoder 不是被硬编码成：

```text
target 第 3 个位置 -> source 第 3 个位置
```

而是根据当前 decoder state 与各 encoder state 的内容表示来判断相关性。

所以不同 decoder steps 可以对同一 source 做不同的 lookup。


## 3. Attention Weights：把“相关性”变成 Source Positions 上的分布

raw alignment scores 还不是最终可直接解释的 attention weights。

Attention 会把一个 decoder step 对所有 `T_src` 个 source positions 的 raw scores 转成一组 normalized coefficients：

$$
\alpha_{u,0},
\alpha_{u,1},
\ldots,
\alpha_{u,T_{src}-1}.
$$

这一组系数满足：

$$
\alpha_{u,j}\ge 0
$$

以及：

$$
\sum_{j=0}^{T_{src}-1}\alpha_{u,j}=1.
$$

因此可以把：

$$
\alpha_u
\in
\mathbb R^{B\times T_{src}}
$$

理解成 decoder step `u` 在 source positions 上的一组 **attention distribution（注意力分布）**。

例如某一步可能是：

```text
source position:    0     1     2     3
attention weight: 0.05  0.10  0.80  0.05
```

直觉上它表示：这个 decoder step 当前主要依赖 source position 2，但仍然保留对其他 positions 的较小关注。

注意两个边界：

1. attention distribution 不要求 one-hot；模型可以同时利用多个 source positions；
2. `α` 是模型内部的 normalized relevance coefficients，不应该自动解释成“人类语义上的真实概率”或绝对可靠的解释。


In [ ]:
import mlx.core as mx

# These are manually chosen attention distributions for intuition only.
# They are NOT computed from model scores in this lesson.
attention_weights = mx.array([
    [0.80, 0.10, 0.05, 0.05],
    [0.10, 0.75, 0.10, 0.05],
    [0.05, 0.10, 0.80, 0.05],
    [0.02, 0.03, 0.05, 0.90],
])  # (T_tgt,T_src) for one illustrative sample

print('attention_weights shape:', attention_weights.shape)
print('row sums:', mx.sum(attention_weights, axis=-1))


## 4. Dynamic Context：同一条 Source，不同 Decoder Steps 可以看不同位置

这正是 fixed-vector Seq2Seq 与 Attention 的关键差别。

Fixed-vector Seq2Seq：

```text
decoder step 0 ─┐
decoder step 1 ─┼─> same fixed source summary s
decoder step 2 ─┤
decoder step 3 ─┘
```

Attention intuition：

```text
decoder step 0 -> attention distribution α_0 -> source states
decoder step 1 -> attention distribution α_1 -> source states
decoder step 2 -> attention distribution α_2 -> source states
...
```

所以 source-side context 不再必须对所有 target positions 完全相同。

每个 decoder step 都会根据自己的 previous decoder state，得到一组新的 source-position weights。

如果把一个样本的所有 decoder steps 堆起来，可以得到一个矩阵：

$$
A
\in
\mathbb R^{T_{tgt}\times T_{src}}.
$$

第 `u` 行就是 decoder step `u` 的 source attention distribution。

下面 heatmap 使用**手工给定**的 weights，只展示“不同 target positions 可以有不同 source focus”这个概念，不代表训练模型的真实输出。


In [ ]:
import matplotlib.pyplot as plt

source_tokens = ['I', 'love', 'coffee', '.']
target_tokens = ['我', '爱', '咖啡', '。']

weights_for_plot = [
    [0.80, 0.10, 0.05, 0.05],
    [0.10, 0.75, 0.10, 0.05],
    [0.05, 0.10, 0.80, 0.05],
    [0.02, 0.03, 0.05, 0.90],
]

fig, ax = plt.subplots(figsize=(6, 4))
image = ax.imshow(weights_for_plot, aspect='auto')

ax.set_xticks(range(len(source_tokens)), source_tokens)
ax.set_yticks(range(len(target_tokens)), target_tokens)
ax.set_xlabel('source position')
ax.set_ylabel('decoder / target step')
ax.set_title('Illustrative dynamic attention distributions')

for u, row in enumerate(weights_for_plot):
    for j, value in enumerate(row):
        ax.text(j, u, f'{value:.2f}', ha='center', va='center')

fig.colorbar(image, ax=ax, label='attention weight')
plt.show()


## 5. Attention 产生的是“每一步自己的 Source Context”，但精确计算留到 10.4

对 decoder step `u`：

```text
encoder states H_enc : (B,T_src,H_enc)
attention weights α_u: (B,T_src)
```

Attention 会利用这组 weights，从整条 encoder state trajectory 中形成当前 step 专用的：

$$
c_u
\in
\mathbb R^{B\times H_{enc}}.
$$

把 `c_u` 称为 **attention context vector（注意力上下文向量）**。

关键区别：

```text
fixed-vector Seq2Seq:
    one source summary s
    reused across decoder steps

attention:
    c_0, c_1, c_2, ...
    decoder step-specific source summaries
```

这里先不写 `c_u` 的精确 weighted-sum 公式，也不解释 raw score 怎样经过 softmax 变成 `α_u`。

这些正是 Step 10.4 的内容。

### Attention 有没有完全消除所有旧问题？

没有。

- recurrent encoder/decoder 的 Time-axis dependency 在这个历史阶段仍可以存在；
- attention 增加了 decoder 对 source states 的直接访问通道，但不意味着 recurrent computation 本身消失；
- attention weights 可以分散，不保证每一步只对应一个 source token；
- source/target positions 的对齐也不一定是一一对应。

所以 10.3 的突破应准确理解为：

> **Decoder 不再只能依赖一个固定 final encoder state，而可以在每个生成步骤，根据当前 decoder state 动态访问整条 encoder state trajectory。**

### Step 10.3 材料边界

- decoder previous state `h_(u-1)^dec` 提供“当前需要什么信息”的 target-side context。
- learned alignment function 比较 decoder state 与各 `h_j^enc`，产生 raw alignment scores `e_(u,j)`。
- normalized attention weights `α_(u,j)` 在 source positions 上非负且和为 1，可理解为 attention distribution。
- 不同 decoder steps 可以得到不同 attention distributions，因此 source context 可以动态变化。
- `c_u` 是当前 decoder step 的 attention context vector，shape 为 `(B,H_enc)`。
- 本轮不推导 softmax，不做 score→softmax→weighted-sum 的精确计算，不使用 Q/K/V 术语。
- 本轮只准备 Step 10.3；不进入 Step 10.4。
